# Phase 2 — PySpark + Parquet | E-commerce Data Engineering

**Goal:** Upgrade the existing Phase 1 project without removing its Pandas/SQLite implementation.

**Architecture:** Raw CSV (Bronze) → PySpark quality checks + joins (Silver) → Parquet datasets → Spark SQL Gold analytics.

Dataset: **100,100** input orders, including **100** duplicate rows and **625** intentionally invalid records. Expected Silver: **99,375** valid orders.

**Important:** Google Colab provides a temporary runtime; output Parquet files disappear when the runtime resets unless saved externally.

## 1. Install PySpark and clone GitHub repo

PySpark is Python's API for Apache Spark. It lets us express distributed DataFrame transformations and SQL queries. We use local Spark on Colab; this teaches the same API but **does not demonstrate a multi-machine Spark cluster**.

In [ ]:
%pip -q install "pyspark>=3.5,<4"

In [ ]:
from pathlib import Path
import subprocess, sys

PROJECT = Path("/content/ecommerce-data-engineering-pipeline")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/mosesrathan19/ecommerce-data-engineering-pipeline.git",
                    str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(PROJECT))
print("Repository:", PROJECT)

## 2. Start Spark and examine Bronze (raw source)

SparkSession is the entry point for DataFrame and SQL operations. We read all **20 order CSV shards** with a defined string schema, plus customer/product dimension files.

In [ ]:
from src.spark_pipeline import spark_session, extract, transform, gold_tables, run
from pyspark.sql import functions as F
spark = spark_session()
print("Spark version:", spark.version)
orders, customers, products = extract(spark, PROJECT / "data/raw")
print("Orders:", orders.count(), "Customers:", customers.count(), "Products:", products.count())
orders.show(5, truncate=False)
orders.printSchema()

## 3. Transform into Silver + quarantine

We cast numeric fields and dates, deduplicate by **order_id**, left-join customers/products, and quarantine orders that fail validation. We avoid dropping invalid records silently.

**What is a Spark transformation?** Operations such as `filter`, `join`, and `withColumn` define computation. Actions such as `count` and `show` cause Spark to execute it.

In [ ]:
unique, silver, rejected = transform(orders, customers, products)
print("Unique orders:", unique.count())
print("Silver accepted:", silver.count())
print("Quarantined:", rejected.count())
silver.select("order_id","product_name","state","quantity","gross_amount","recognized_revenue").show(5, truncate=False)
rejected.select("order_id","reject_reasons").show(5, truncate=False)

## 4. Save Parquet datasets

**Parquet** is a columnar file format commonly used in analytical pipelines. We write the clean Silver fact table and the quarantine dataset, and then aggregate Gold tables by month, category, and state. These are **overwrite/full-refresh** outputs; incremental loads come in Phase 3.

In [ ]:
OUTPUT = PROJECT / "outputs" / "spark"
quality, gold = run(spark, PROJECT / "data/raw", OUTPUT)
print("Quality report:", quality)
assert quality == {
    "raw_rows": 100100, "unique_orders": 100000,
    "duplicates_removed": 100, "accepted_rows": 99375,
    "rejected_rows": 625, "all_rows_accounted_for": True,
}
print("Parquet files written under", OUTPUT)

## 5. Read Silver Parquet back and query with SQL

Spark can read the saved Parquet folder and register it as a temporary SQL view. Revenue is recognized only for `completed` orders in this simplified example.

In [ ]:
silver_parquet = spark.read.parquet(str(OUTPUT / "silver" / "orders"))
silver_parquet.createOrReplaceTempView("silver_orders")
spark.sql("""
SELECT category,
       COUNT(*) AS orders,
       ROUND(SUM(recognized_revenue), 2) AS revenue
FROM silver_orders
GROUP BY category
ORDER BY revenue DESC
""").show(truncate=False)

## 6. Inspect Gold and plot a chart

Aggregated Gold data is ready for business intelligence. We use `toPandas()` only **after aggregation**, when the result is small enough to fit in memory.

In [ ]:
import matplotlib.pyplot as plt
category = spark.read.parquet(str(OUTPUT / "gold" / "category_revenue"))
category.show(truncate=False)
pdf = category.orderBy(F.desc("revenue")).toPandas()
plt.figure(figsize=(9,4))
plt.bar(pdf["category"], pdf["revenue"].astype(float))
plt.xlabel("Product category")
plt.ylabel("Recognized revenue")
plt.title("Gold layer: Revenue by product category")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Verify outputs and reflect on production improvements

Run the validation checks below. **Why this matters:** a successful job is not enough; data completeness and quality must be measured.

Phase 3 enhancements: partition Parquet by date, incremental processing, upsert strategies, late-arriving records, and checkpointing. Phase 4: scheduling and monitoring.

In [ ]:
assert silver_parquet.count() == 99375
assert spark.read.parquet(str(OUTPUT/"quarantine"/"orders")).count() == 625
assert spark.read.parquet(str(OUTPUT/"gold"/"monthly_revenue")).count() == 12
assert silver_parquet.select("order_id").distinct().count() == 99375
print("All Phase 2 checks passed ✅")
print("Silver schema:")
silver_parquet.printSchema()

## Optional: Persist outputs to Google Drive

For Colab, the local `/content` folder is temporary. To save results beyond this session, mount Drive and copy the `outputs/spark` directory there. Do not commit generated Parquet data to GitHub unless you have a deliberate reason.

In [ ]:
# Uncomment to save results to Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copytree(OUTPUT, "/content/drive/MyDrive/ecommerce-spark-outputs", dirs_exist_ok=True)